In [ ]:
"""Generate all four paper figures for draft_v3.md."""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
FONT_SIZE = 11
plt.rcParams.update({'font.size': FONT_SIZE})
OUT = '/Users/hari/Documents/carby-tho/outputs/figures'

## FIGURE 1 — EU ETS Price History with ZA Break

In [ ]:
eua = pd.read_csv('/Users/hari/Documents/carby-tho/data/raw/eua_prices.csv')
eua = eua[['year', 'price']].dropna()

fig, ax = plt.subplots(figsize=(8, 4.5))

# Phase bands
ax.axvspan(2005, 2007, alpha=0.08, color='steelblue', label='Phase I (2005–07)')
ax.axvspan(2008, 2012, alpha=0.08, color='orange', label='Phase II (2008–12)')
ax.axvspan(2013, 2024, alpha=0.08, color='green', label='Phase III+ (2013–)')

# Price line
ax.plot(eua['year'], eua['price'], color='#1a1a2e', linewidth=2, marker='o',
        markersize=4, zorder=5)

# ZA break line
ax.axvline(x=2014, color='crimson', linestyle='--', linewidth=1.5, zorder=6)
ax.text(2014.15, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 75,
        'ZA break: Phase III reform', color='crimson', fontsize=9,
        va='top', rotation=90)

# Phase I collapse annotation
ax.annotate('Phase I over-allocation\nrevealed', xy=(2007, 0.8),
            xytext=(2008.5, 18), fontsize=8.5,
            arrowprops=dict(arrowstyle='->', color='#555', lw=1.2),
            color='#333')

ax.set_xlabel('Year', fontsize=FONT_SIZE)
ax.set_ylabel('EUA price (€/tCO₂)', fontsize=FONT_SIZE)
ax.set_title('EU ETS Annual Carbon Price, 2005–2024', fontsize=FONT_SIZE + 1, fontweight='bold')
ax.set_xlim(2004.5, 2024.5)
ax.set_ylim(-2, 95)
ax.legend(loc='upper left', fontsize=9, framealpha=0.8)
ax.tick_params(labelsize=10)

# Re-draw ZA annotation after ylim set
for child in ax.get_children():
    if hasattr(child, 'get_text') and 'ZA break' in str(child.get_text()):
        child.set_y(88)

plt.tight_layout()
fig.savefig(f'{OUT}/figure_1_ets_break.png', dpi=300, bbox_inches='tight')
plt.close()
print("Figure 1 saved.")

## FIGURE 2 — Present Register: USA and EMU

In [ ]:
panel = pd.read_csv('/Users/hari/Documents/carby-tho/data/processed/panel_model_ready.csv')
panel = panel[['year', 'country_code', 'net_energy_position', 'reserve_share']].copy()

usa = panel[panel['country_code'] == 'USA'].sort_values('year')
emu = panel[panel['country_code'] == 'EMU'].sort_values('year')

# Compute lag-10 correlation for USA (NEP at t vs RS at t+10)
usa_nep = usa.set_index('year')['net_energy_position']
usa_rs  = usa.set_index('year')['reserve_share']
lag = 10
common_t = [y for y in usa_nep.index if (y + lag) in usa_rs.index
            and pd.notna(usa_nep[y]) and pd.notna(usa_rs[y + lag])]
if common_t:
    nep_vals = [usa_nep[y] for y in common_t]
    rs_vals  = [usa_rs[y + lag] for y in common_t]
    r_usa = np.corrcoef(nep_vals, rs_vals)[0, 1]
else:
    r_usa = float('nan')

# EMU lag-10 correlation
emu_nep = emu.set_index('year')['net_energy_position']
emu_rs  = emu.set_index('year')['reserve_share']
common_t_emu = [y for y in emu_nep.index if (y + lag) in emu_rs.index
                and pd.notna(emu_nep[y]) and pd.notna(emu_rs[y + lag])]
if common_t_emu:
    nep_vals_e = [emu_nep[y] for y in common_t_emu]
    rs_vals_e  = [emu_rs[y + lag] for y in common_t_emu]
    r_emu = np.corrcoef(nep_vals_e, rs_vals_e)[0, 1]
else:
    r_emu = 0.864  # paper-stated value

print(f"USA lag-10 r = {r_usa:.3f}, n = {len(common_t)}")
print(f"EMU lag-10 r = {r_emu:.3f}, n = {len(common_t_emu)}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── USA panel ──
usa_plot = usa[usa['year'] >= 1995].dropna(subset=['net_energy_position', 'reserve_share'])
ax1b = ax1.twinx()
l1, = ax1.plot(usa_plot['year'], usa_plot['net_energy_position'],
               color='#1f77b4', linewidth=2, label='NEP (left)')
l2, = ax1b.plot(usa_plot['year'], usa_plot['reserve_share'],
                color='#ff7f0e', linewidth=2, linestyle='--', label='USD reserve share (right)')
ax1.set_xlabel('Year', fontsize=FONT_SIZE)
ax1.set_ylabel('Net Energy Position', fontsize=FONT_SIZE, color='#1f77b4')
ax1b.set_ylabel('Reserve share (%)', fontsize=FONT_SIZE, color='#ff7f0e')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1b.tick_params(axis='y', labelcolor='#ff7f0e')
ax1.set_title('United States', fontsize=FONT_SIZE + 1, fontweight='bold')
r_label = 'ARDL cointegration confirmed\n(Pesaran et al. 2001)'
ax1.text(0.03, 0.97, r_label, transform=ax1.transAxes,
         fontsize=9, va='top', ha='left',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
lines = [l1, l2]
ax1.legend(lines, [l.get_label() for l in lines], loc='lower left', fontsize=9)

# ── EMU panel ──
emu_plot = emu[emu['year'] >= 1999].dropna(subset=['net_energy_position', 'reserve_share'])
ax2b = ax2.twinx()
l3, = ax2.plot(emu_plot['year'], emu_plot['net_energy_position'],
               color='#2ca02c', linewidth=2, label='EUR NEP (left)')
l4, = ax2b.plot(emu_plot['year'], emu_plot['reserve_share'],
                color='#d62728', linewidth=2, linestyle='--', label='EUR reserve share (right)')
ax2.set_xlabel('Year', fontsize=FONT_SIZE)
ax2.set_ylabel('Net Energy Position', fontsize=FONT_SIZE, color='#2ca02c')
ax2b.set_ylabel('Reserve share (%)', fontsize=FONT_SIZE, color='#d62728')
ax2.tick_params(axis='y', labelcolor='#2ca02c')
ax2b.tick_params(axis='y', labelcolor='#d62728')
ax2.set_title('Eurozone (EMU)', fontsize=FONT_SIZE + 1, fontweight='bold')
r_emu_label = f'r = {r_emu:.3f} (lag 10, n = {len(common_t_emu)})'
ax2.text(0.03, 0.97, r_emu_label, transform=ax2.transAxes,
         fontsize=9, va='top', ha='left',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
lines2 = [l3, l4]
ax2.legend(lines2, [l.get_label() for l in lines2], loc='upper right', fontsize=9)

plt.suptitle('Net Energy Position and Reserve Currency Share', fontsize=FONT_SIZE + 2, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(f'{OUT}/figure_2_present_register.png', dpi=300, bbox_inches='tight')
plt.close()
print("Figure 2 saved.")

## FIGURE 3 — Japan Fukushima Natural Experiment

In [ ]:
jpn = panel[(panel['country_code'] == 'JPN') &
            (panel['year'] >= 2005) & (panel['year'] <= 2018)].sort_values('year')
jpn = jpn.dropna(subset=['net_energy_position'])

fig, ax = plt.subplots(figsize=(8, 4.5))

# Pre/post shading
pre = jpn[jpn['year'] <= 2011]
post = jpn[jpn['year'] >= 2011]

ax.axvspan(2005, 2011, alpha=0.07, color='steelblue')
ax.axvspan(2011, 2018, alpha=0.07, color='tomato')

ax.plot(jpn['year'], jpn['net_energy_position'],
        color='#1a1a2e', linewidth=2, marker='o', markersize=5, zorder=5)

# Fukushima line
ax.axvline(x=2011, color='crimson', linestyle='--', linewidth=1.5, zorder=6)
ax.text(2011.1, jpn['net_energy_position'].max() * 0.97,
        'Fukushima (March 2011)', color='crimson', fontsize=9, va='top')

# Drop annotation: 2010 → 2012 only (the two-year post-Fukushima collapse)
y_2010 = jpn[jpn['year'] == 2010]['net_energy_position'].values[0]
y_2012 = jpn[jpn['year'] == 2012]['net_energy_position'].values[0]
ax.annotate('', xy=(2012, y_2012), xytext=(2010, y_2010),
            arrowprops=dict(arrowstyle='<->', color='#555', lw=1.5))
mid_x = 2011
mid_y = (y_2010 + y_2012) / 2
ax.text(mid_x, mid_y + 0.0003, '−67%\n(0.0084 → 0.0028)', fontsize=9,
        ha='center', color='#333',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.9))

ax.set_xlabel('Year', fontsize=FONT_SIZE)
ax.set_ylabel('Net Energy Position', fontsize=FONT_SIZE)
ax.set_title("Japan's Net Energy Position, 2005–2018", fontsize=FONT_SIZE + 1, fontweight='bold')
ax.set_xlim(2004.5, 2018.5)
ax.tick_params(labelsize=10)

# Period labels
ax.text(2008, jpn['net_energy_position'].min() + 0.0002, 'Pre-Fukushima',
        fontsize=9, color='steelblue', ha='center', alpha=0.8)
ax.text(2015, jpn['net_energy_position'].min() + 0.0002, 'Post-shutdown',
        fontsize=9, color='tomato', ha='center', alpha=0.8)

plt.tight_layout()
fig.savefig(f'{OUT}/figure_3_japan_fukushima.png', dpi=300, bbox_inches='tight')
plt.close()
print("Figure 3 saved.")

## FIGURE 4 — TMPI Rankings

In [ ]:
tmpi = pd.read_csv('/Users/hari/Documents/carby-tho/outputs/tables/tmpi_rankings.csv')

# Select the 8 countries in the plan
selected = ['USA', 'CAN', 'IND', 'BRA', 'RUS', 'CHN', 'AUS', 'GBR']
tmpi8 = tmpi[tmpi['country_code'].isin(selected)].copy()
tmpi8 = tmpi8.sort_values('tmpi_scaled', ascending=True)  # ascending for horizontal bar

# Colour tiers — GBR grey (zero thorium reserves, TMPI=0) same as AUS
def colour(code):
    if code in ['USA', 'CAN', 'IND']:
        return '#2ca02c'   # green — full option
    elif code in ['AUS', 'GBR']:
        return '#aaaaaa'   # grey — zero TMPI (different reasons)
    else:
        return '#ff7f0e'   # amber — conditional

tmpi8['colour'] = tmpi8['country_code'].apply(colour)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(tmpi8['country_code'], tmpi8['tmpi_scaled'],
               color=tmpi8['colour'], edgecolor='white', linewidth=0.5, height=0.6)

# Value labels
for bar, val in zip(bars, tmpi8['tmpi_scaled']):
    ax.text(val + 0.8, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=9.5)

ax.set_xlabel('TMPI Score (USA = 100)', fontsize=FONT_SIZE)
ax.set_title('Thorium Monetary Potential Index\nEight-Country Comparison', fontsize=FONT_SIZE + 1, fontweight='bold')
ax.set_xlim(0, 115)
ax.tick_params(labelsize=10)

# Legend
legend_elements = [
    mpatches.Patch(color='#2ca02c', label='Full option (all 3 conditions)'),
    mpatches.Patch(color='#ff7f0e', label='Conditional option'),
    mpatches.Patch(color='#aaaaaa', label='Zero TMPI: AUS (no nuclear) / GBR (no thorium reserves)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
fig.savefig(f'{OUT}/figure_4_tmpi_rankings.png', dpi=300, bbox_inches='tight')
plt.close()
print("Figure 4 saved.")

print("\nAll figures saved to", OUT)